```python
"""
01_pipeline_setup.py
────────────────────
distilgpt2 hook + activation extraction + buffer for SAE training.

Run as a Kaggle notebook cell-by-cell, or import the classes in notebook 02.

Section map
  1. Config
  2. Model & hook
  3. Token streaming + sequence packing
  4. Normalization
  5. ActivationBuffer   ← online streaming, same session
  6. ShardedBuffer      ← offline, load pre-saved shards in a separate session
  7. Extraction helpers
  8. Orchestration (main)
"""
```

In [ ]:
# ── 1. Imports & config ───────────────────────────────────────────────────────
import os
import math
import torch
import numpy as np
from pathlib import Path
from itertools import islice
from typing import Iterator, Optional

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
LAYER_IDX     = 3            # layer 3 of 6 in distilgpt2
SEQ_LEN       = 128          # contiguous, non-overlapping context windows
D_MODEL       = 768          # distilgpt2 hidden dimension
BUFFER_SIZE   = 2_000_000    # activations held in CPU RAM at once (≈ 3 GB fp16)
EXTRACT_BATCH = 64           # sequences per LM forward pass
DEBUG_TOKENS  = 200_000      # 0.2 % of OWT for the debug / normalizer-fit pass
FULL_TOKENS   = 10_000_000   # ≈ 1 % of OWT for full extraction
SHARD_SIZE    = 1_000_000    # rows per saved shard (≈ 1.5 GB fp16 each)
SAE_BATCH     = 4_096

OUT_DIR = Path("/kaggle/working/activations")
OUT_DIR.mkdir(parents=True, exist_ok=True)



In [ ]:
# ── 2. Model & hook ───────────────────────────────────────────────────────────
def load_frozen_model(device: str = DEVICE):
    """
    Load distilgpt2 with all parameters frozen and in eval mode.
    Keep weights in fp32 for numerical correctness; activations are cast to
    fp16 in the hook to save RAM during extraction.
    """
    tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

    model = AutoModelForCausalLM.from_pretrained(
        "distilgpt2", torch_dtype=torch.float32
    ).to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    return model, tokenizer


class ActivationHook:
    """
    Forward hook on distilgpt2's transformer block at LAYER_IDX.

    distilgpt2 block forward() returns a tuple:
        (hidden_states, present_key_value, [attentions])
    hidden_states has shape (batch, seq_len, d_model) — that is output[0].

    The hook stores captured tensors internally; call .pop_all() after each
    forward pass to retrieve and clear them. Never hold references across
    many forward passes without clearing, or memory will grow unbounded.
    """

    def __init__(self):
        self._buf: list[torch.Tensor] = []
        self._handle = None

    def register(self, model: AutoModelForCausalLM, layer_idx: int = LAYER_IDX):
        # model.transformer.h is the ModuleList of GPT-2 Block instances
        target = model.transformer.h[layer_idx]
        self._handle = target.register_forward_hook(self._fn)
        return self  # fluent

    def _fn(self, module, input, output):
        # Detach immediately: we do not want gradients, and we do not want the
        # computation graph keeping every intermediate tensor alive.
        # Cast to fp16 here; saves ≈ 50% RAM during a long extraction run.
        self._buf.append(output[0].detach().cpu().to(torch.float16))

    def pop_all(self) -> torch.Tensor:
        """Return (batch*n_calls, seq_len, d_model) fp16 and clear the buffer."""
        if not self._buf:
            return torch.empty(0, SEQ_LEN, D_MODEL, dtype=torch.float16)
        out = torch.cat(self._buf, dim=0)
        self._buf.clear()
        return out

    def remove(self):
        if self._handle is not None:
            self._handle.remove()
            self._handle = None

    # context-manager support so the hook is always cleaned up
    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.remove()



In [ ]:
# ── 3. Token streaming & sequence packing ────────────────────────────────────
def _owt_token_stream(tokenizer, max_tokens: int) -> Iterator[torch.Tensor]:
    """
    Yields non-overlapping 128-token windows from OpenWebText (streaming).

    Documents are concatenated with an EOS separator so the model sees a
    realistic distribution of document boundaries. Tails shorter than SEQ_LEN
    are discarded rather than padded — the assignment requires contiguous
    sequences with no padding.
    """
    ds = load_dataset(
        "Skylion007/openwebtext",
        split="train",
        streaming=True,
    )

    carry = torch.empty(0, dtype=torch.long)
    yielded = 0

    for ex in ds:
        if yielded >= max_tokens:
            break

        ids = tokenizer.encode(
            ex["text"], add_special_tokens=False, return_tensors="pt"
        ).squeeze(0)
        # separator between documents
        ids = torch.cat([carry, ids, torch.tensor([tokenizer.eos_token_id])])

        n_windows = len(ids) // SEQ_LEN
        for i in range(n_windows):
            if yielded >= max_tokens:
                break
            yield ids[i * SEQ_LEN : (i + 1) * SEQ_LEN]
            yielded += SEQ_LEN

        carry = ids[n_windows * SEQ_LEN :]   # keep the partial tail


def _batch_windows(
    stream: Iterator[torch.Tensor], batch_size: int
) -> Iterator[torch.Tensor]:
    """Collates individual 128-token windows into (batch_size, SEQ_LEN) batches."""
    while True:
        chunk = list(islice(stream, batch_size))
        if not chunk:
            break
        yield torch.stack(chunk)   # (B, 128)



In [ ]:
# ── 4. Normalization ──────────────────────────────────────────────────────────
class ActivationNormalizer:
    """
    Per-dimension (channel-wise) z-score normalizer.

    Fit on the debug pass (~200k tokens); save and reuse for the full extraction
    and at eval time. Consistent normalization is critical: the SAE learns a
    dictionary in the normalized space, so test-time patching must use the same
    mean/std.

    Why per-dimension?  distilgpt2 layer-3 output dimensions have very different
    scales — some are near-zero throughout the dataset, others span ±10+.
    Global (scalar) normalization leaves that structure intact, making the SAE
    learn an uneven dictionary. Per-dimension normalization is what Anthropic and
    Nanda's public SAE implementations both use.
    """

    def __init__(self):
        self.mean: Optional[torch.Tensor] = None
        self.std:  Optional[torch.Tensor] = None

    def fit(self, acts: torch.Tensor, eps: float = 1e-6):
        """acts: (N, D_MODEL), any dtype."""
        a = acts.float()
        self.mean = a.mean(dim=0)                    # (D_MODEL,)
        self.std  = a.std(dim=0).clamp(min=eps)      # (D_MODEL,)
        print(
            f"  normalizer: mean.norm={self.mean.norm():.3f}  "
            f"std.mean={self.std.mean():.4f}  "
            f"std.min={self.std.min():.6f}"
        )

    def __call__(self, acts: torch.Tensor) -> torch.Tensor:
        """Return normalized activations in the same dtype as input."""
        dtype = acts.dtype
        a = acts.float()
        out = (a - self.mean.to(a.device)) / self.std.to(a.device)
        return out.to(dtype)

    def inverse(self, acts: torch.Tensor) -> torch.Tensor:
        dtype = acts.dtype
        a = acts.float()
        out = a * self.std.to(a.device) + self.mean.to(a.device)
        return out.to(dtype)

    def save(self, path: Path):
        torch.save({"mean": self.mean, "std": self.std}, path)
        print(f"  normalizer saved → {path}")

    @classmethod
    def load(cls, path: Path) -> "ActivationNormalizer":
        n = cls()
        ckpt = torch.load(path, map_location="cpu")
        n.mean, n.std = ckpt["mean"], ckpt["std"]
        return n



In [ ]:
# ── 5. ActivationBuffer (online / same-session streaming) ────────────────────
class ActivationBuffer:
    """
    Fixed-capacity CPU-RAM buffer for SAE training.

    Fills from an extraction generator, shuffles in-place, yields GPU batches.
    When the current buffer is consumed it refills automatically from the same
    generator, so the SAE trainer just calls next(buffer) in a loop.

    Memory layout
    ─────────────
    self._buf is a pre-allocated (BUFFER_SIZE, D_MODEL) fp16 tensor sitting in
    CPU RAM (~3 GB at the defaults). Batches are cast to fp32 on the GPU only
    when the SAE actually needs them, minimising peak VRAM usage.

    Why shuffle within a window rather than globally?
    ─────────────────────────────────────────────────
    Storing all 80 M activations would require ~123 GB. Shuffling within a 2 M
    window is the standard compromise used by Anthropic and Nanda: it breaks
    the strong temporal correlation within a document while fitting comfortably
    in RAM. 2 M >> typical SAE batch (4096), so each batch is effectively i.i.d.
    """

    def __init__(
        self,
        generator: Iterator[torch.Tensor],
        normalizer: ActivationNormalizer,
        capacity: int = BUFFER_SIZE,
        sae_batch_size: int = SAE_BATCH,
        device: str = DEVICE,
    ):
        self._gen      = generator
        self._norm     = normalizer
        self._cap      = capacity
        self._bsz      = sae_batch_size
        self._device   = device

        # Pre-allocate once; writing into it is done via slice assignment
        self._buf     = torch.empty(capacity, D_MODEL, dtype=torch.float16)
        self._n       = 0     # rows currently valid in _buf
        self._ptr     = 0     # next row to hand out

        self._fill()

    # ── internal ──────────────────────────────────────────────────────────────
    def _fill(self):
        write = 0
        for chunk in self._gen:
            # chunk is fp16, (n, D_MODEL), already on CPU
            normed = self._norm(chunk)               # still fp16
            end    = min(write + len(normed), self._cap)
            n      = end - write
            self._buf[write:end] = normed[:n]
            write = end
            if write >= self._cap:
                # stash any overflow from this chunk back into the generator
                # by wrapping it as a one-step iterator prepended to self._gen
                if n < len(normed):
                    leftover = normed[n:]
                    self._gen = _prepend(leftover, self._gen)
                break

        self._n = write
        if self._n == 0:
            raise StopIteration("Activation generator exhausted")

        # Shuffle the valid portion in-place (avoids a copy)
        perm = torch.randperm(self._n)
        self._buf[:self._n] = self._buf[perm]
        self._ptr = 0
        print(f"  buffer: {self._n:,} activations loaded & shuffled")

    # ── public ────────────────────────────────────────────────────────────────
    def __iter__(self):
        return self

    def __next__(self) -> torch.Tensor:
        """Returns one SAE batch, fp32, on GPU."""
        if self._ptr + self._bsz > self._n:
            self._fill()
        batch = self._buf[self._ptr : self._ptr + self._bsz]
        self._ptr += self._bsz
        # non_blocking=True overlaps the H→D copy with GPU computation
        return batch.to(self._device, dtype=torch.float32, non_blocking=True)

    @property
    def n_remaining(self) -> int:
        return self._n - self._ptr


def _prepend(tensor: torch.Tensor, gen: Iterator) -> Iterator:
    """Utility: yield a single tensor before resuming a generator."""
    yield tensor
    yield from gen



In [ ]:
# ── 6. ShardedBuffer (cross-session, load pre-saved shards) ──────────────────
class ShardedActivationBuffer:
    """
    Drop-in replacement for ActivationBuffer when activations were saved to
    disk in a previous Kaggle session (Notebook 01 → Notebook 02).

    Loads one shard at a time, shuffles it, and yields SAE batches.  Cycles
    through all shards indefinitely (wraps around), which is fine because the
    SAE only needs ~100 k steps × 4096 batch ≈ 400 M activation rows total —
    less than the 80 M you extracted (each row seen ≈ 5 times on average).
    """

    def __init__(
        self,
        shard_dir: Path,
        sae_batch_size: int = SAE_BATCH,
        device: str = DEVICE,
    ):
        self._paths = sorted(shard_dir.glob("shard_*.pt"))
        assert self._paths, f"No shards found in {shard_dir}"
        self._bsz    = sae_batch_size
        self._device = device
        self._idx    = 0
        self._buf: Optional[torch.Tensor] = None
        self._ptr    = 0
        self._load_next()

    def _load_next(self):
        path = self._paths[self._idx % len(self._paths)]
        self._idx += 1
        buf = torch.load(path, map_location="cpu")   # fp16, (shard_size, D_MODEL)
        # shuffle
        buf = buf[torch.randperm(len(buf))]
        self._buf = buf
        self._ptr = 0
        print(f"  shard loaded: {path.name}  ({len(self._buf):,} rows)")

    def __iter__(self):
        return self

    def __next__(self) -> torch.Tensor:
        if self._ptr + self._bsz > len(self._buf):
            self._load_next()
        batch = self._buf[self._ptr : self._ptr + self._bsz]
        self._ptr += self._bsz
        return batch.to(self._device, dtype=torch.float32, non_blocking=True)



In [ ]:
# ── 7. Extraction helpers ────────────────────────────────────────────────────
@torch.no_grad()
def _extraction_gen(
    model,
    tokenizer,
    hook: ActivationHook,
    max_tokens: int,
    tag: str = "",
) -> Iterator[torch.Tensor]:
    """
    Generator: runs distilgpt2 forward passes and yields flat fp16 activation
    chunks of shape (n, D_MODEL).  One forward pass produces EXTRACT_BATCH×SEQ_LEN
    rows.  This is an internal helper; callers feed it into ActivationBuffer or
    the shard writer.
    """
    token_stream = _owt_token_stream(tokenizer, max_tokens)
    batch_stream = _batch_windows(token_stream, EXTRACT_BATCH)

    seen = 0
    log_every = max(1, max_tokens // (SEQ_LEN * 20))   # log ~20 times total

    for input_ids in batch_stream:
        model(input_ids.to(DEVICE))
        acts = hook.pop_all().view(-1, D_MODEL)    # (B*128, 768), fp16, CPU
        seen += len(acts)
        if (seen // len(acts)) % log_every == 0:
            pct = 100 * seen / max_tokens
            print(f"  [{tag}] {seen:>10,} / {max_tokens:,} tokens  ({pct:.1f}%)")
        yield acts


def save_shards(
    model,
    tokenizer,
    hook: ActivationHook,
    normalizer: ActivationNormalizer,
    max_tokens: int,
    out_dir: Path,
    shard_size: int = SHARD_SIZE,
):
    """
    Extract activations and write them to disk as fp16 shards.
    Use this when you want to train the SAE in a separate Kaggle session so you
    don't have to re-run the expensive LM forward passes.

    Each shard:  shard_NNNN.pt  →  tensor of shape (shard_size, D_MODEL), fp16
    """
    gen   = _extraction_gen(model, tokenizer, hook, max_tokens, tag="shard")
    acc   = []
    n_acc = 0
    idx   = 0

    for chunk in gen:
        normed = normalizer(chunk)   # fp16
        acc.append(normed)
        n_acc += len(normed)

        while n_acc >= shard_size:
            combined = torch.cat(acc, dim=0)
            shard    = combined[:shard_size]
            leftover = combined[shard_size:]
            path = out_dir / f"shard_{idx:04d}.pt"
            torch.save(shard, path)
            size_gb = path.stat().st_size / 1e9
            print(f"  saved {path.name}  ({shard_size:,} rows  {size_gb:.2f} GB)")
            idx  += 1
            acc   = [leftover] if len(leftover) else []
            n_acc = len(leftover)

    # final partial shard
    if acc:
        combined = torch.cat(acc, dim=0)
        path = out_dir / f"shard_{idx:04d}.pt"
        torch.save(combined, path)
        print(f"  saved final {path.name}  ({len(combined):,} rows)")

    print(f"  extraction complete: {idx + 1} shards → {out_dir}")



In [ ]:
# ── 8. Orchestration ──────────────────────────────────────────────────────────
def run_debug_pass(model, tokenizer, hook, out_dir: Path):
    """
    Runs the 0.2 % extraction pass, fits and saves the normalizer, and returns
    both the raw debug activations (for inspection) and the fitted normalizer.
    """
    print("=" * 60)
    print(f"DEBUG PASS  ({DEBUG_TOKENS:,} tokens)")
    print("=" * 60)

    gen       = _extraction_gen(model, tokenizer, hook, DEBUG_TOKENS, tag="debug")
    debug_raw = torch.cat(list(gen), dim=0)   # safe: ≈ 200k × 768 × 2 B ≈ 300 MB

    print(f"\n  shape : {debug_raw.shape}")
    print(f"  dtype : {debug_raw.dtype}")
    a = debug_raw.float()
    print(f"  mean  : {a.mean():.4f}   std : {a.std():.4f}")
    print(f"  min   : {a.min():.4f}   max : {a.max():.4f}")

    normalizer = ActivationNormalizer()
    normalizer.fit(debug_raw)
    normalizer.save(out_dir / "normalizer.pt")

    # quick sanity check post-normalization
    normed = normalizer(debug_raw.float())
    print(f"\n  post-norm — mean: {normed.mean():.4f}  std: {normed.std():.4f}")
    assert normed.shape == (len(debug_raw), D_MODEL)

    return debug_raw, normalizer


def build_streaming_buffer(model, tokenizer, hook, normalizer) -> ActivationBuffer:
    """
    Build an ActivationBuffer backed by a live extraction generator.
    Use this when training the SAE in the same session as extraction.
    """
    print("\n" + "=" * 60)
    print(f"BUILDING STREAMING BUFFER  ({FULL_TOKENS:,} tokens)")
    print("=" * 60)
    gen = _extraction_gen(model, tokenizer, hook, FULL_TOKENS, tag="full")
    buf = ActivationBuffer(gen, normalizer)

    # smoke test
    sample = next(buf)
    print(f"\n  first SAE batch — shape: {sample.shape}  device: {sample.device}")
    print(f"  mean: {sample.mean():.4f}  std: {sample.std():.4f}")
    assert sample.shape == (SAE_BATCH, D_MODEL)
    assert sample.device.type == DEVICE
    print("  buffer OK ✓")
    return buf

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    model, tokenizer = load_frozen_model()
    with ActivationHook().register(model) as hook:
        _, normalizer = run_debug_pass(model, tokenizer, hook, OUT_DIR)
        save_shards(model, tokenizer, hook, normalizer,
                    max_tokens=FULL_TOKENS, out_dir=OUT_DIR)